# Computaform South African Horse Racing PDF Extraction

This notebook runs the local extractor for `HOLLYWOODBETS GREYVILLE@2026.05.13.pdf` and writes JSON, Excel, CSV, and Markdown report outputs under `outputs/`.

In [ ]:
# Ensure notebook dependencies are available in the active kernel.
import importlib.util
import subprocess
import sys

packages = {
    "pdfplumber": "pdfplumber",
    "pandas": "pandas",
    "openpyxl": "openpyxl",
}

for package, module in packages.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

print("Dependencies ready")

In [ ]:
# Run the structured extraction.
from pathlib import Path
import json
import sys

project_dir = Path.cwd()
if not (project_dir / "extract_computaform.py").exists():
    project_dir = Path(r"c:\Users\shaun\Downloads\VSCode\Horse Racing\Compu Form")

if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

from extract_computaform import DEFAULT_OUTPUT_DIR, DEFAULT_PDF, extract

data, paths = extract(DEFAULT_PDF, DEFAULT_OUTPUT_DIR)

summary = {
    "source_pdf": data["meeting"]["source_pdf"],
    "pages_processed": data["validation"]["pages_processed"],
    "races_found": data["validation"]["races_found"],
    "runners_found_by_race": data["validation"]["runners_found_by_race"],
    "outputs": {key: str(path) for key, path in paths.items()},
}

print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
# Quick spreadsheet-style previews.
import pandas as pd

races = pd.DataFrame(data["races"])
runners = pd.DataFrame([
    {
        "race_number": runner["race_number"],
        "horse_number": runner["horse_number"],
        "horse_name": runner["horse_name"],
        "draw": runner["draw"],
        "jockey": runner["jockey"],
        "trainer": runner["trainer"],
        "forecast_price": runner["forecast_price"],
        "computaform_rating": runner["computaform_rating"],
        "speed_rating": runner["speed_rating"],
        "past_runs": len(runner["past_runs"]),
    }
    for runner in data["runners"]
])

display(races[["race_number", "race_time", "race_name", "distance", "stake"]])
display(runners.head(20))
display(pd.DataFrame(data["validation"]["runners_found_by_race"]))